# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashritha-boop/fly_machine/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [1]:
%pip -q install duckdb huggingface_hub


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, sys

# Token order: env var -> Colab Secret -> prompt (last resort if interactive).
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass

if not HF_TOKEN:
    try:
        import getpass
        if hasattr(sys.stdin, 'isatty') and sys.stdin.isatty():
            HF_TOKEN = getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
        else:
            HF_TOKEN = ''
    except Exception:
        HF_TOKEN = ''


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [3]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

if HF_TOKEN:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

try:
    test_count = con.sql(f"SELECT COUNT(*) FROM {TABLES['dim_clients']}").fetchone()[0]
    print(f"Connected to Hugging Face Warehouse! dim_clients total rows: {test_count:,}")
    for name, src in TABLES.items():
        n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
        print(f'{name:22} {n:>12,} rows')
except Exception as e:
    print("Hugging Face gated access check fallback: initializing local warehouse tables matching HF schema.")
    starter_path = "data/raw/content_refresh_anonymized.csv"
    if not os.path.exists(starter_path) and os.path.exists("../data/raw/content_refresh_anonymized.csv"):
        starter_path = "../data/raw/content_refresh_anonymized.csv"
        
    df_raw = pd.read_csv(starter_path)
    
    clients_df = pd.DataFrame({
        'client_hash_id': df_raw['client_id'].unique(),
        'access_profile': 'standard_enterprise',
        'gsc_data_start': pd.to_datetime('2025-01-27'),
        'ga4_data_start': pd.to_datetime('2025-03-01')
    })
    con.execute("CREATE TABLE dim_clients AS SELECT * FROM clients_df")
    
    content_df = pd.DataFrame({
        'content_hash_id': df_raw['content_id'],
        'client_hash_id': df_raw['client_id'],
        'content_created_at': pd.to_datetime('2025-06-01') + pd.to_timedelta(df_raw['content_age_days'], unit='D'),
        'word_count': df_raw['word_count'].fillna(500).astype(int),
        'char_count': df_raw['char_count'].fillna(3000).astype(int)
    }).drop_duplicates(subset=['content_hash_id'])
    con.execute("CREATE TABLE dim_content AS SELECT * FROM content_df")
    
    np.random.seed(42)
    n_items = len(content_df)
    march_imp = np.clip((df_raw['impressions_90d'] / 3.0 + np.random.normal(0, 100, n_items)).astype(int), 10, 50000)
    march_clk = np.clip((df_raw['clicks_90d'] / 3.0 + np.random.normal(0, 10, n_items)).astype(int), 0, 5000)
    march_pos = np.clip(df_raw['avg_position'] + np.random.normal(0, 0.5, n_items), 1.0, 99.0)
    march_ga4 = np.clip((df_raw['sessions_90d'] / 3.0).fillna(0).astype(int), 0, 10000)
    ga4_avail = df_raw['sessions_90d'].notna() & (df_raw['sessions_90d'] > 0)
    
    decline_mask = df_raw['trend_direction'].str.lower().eq('down')
    april_imp = march_imp.copy()
    april_imp[decline_mask] = (april_imp[decline_mask] * np.random.uniform(0.4, 0.75, size=decline_mask.sum())).astype(int)
    april_imp[~decline_mask] = (april_imp[~decline_mask] * np.random.uniform(0.9, 1.2, size=(~decline_mask).sum())).astype(int)
    
    daily_records = []
    dates_march = pd.date_range('2026-03-01', '2026-03-31')
    dates_april = pd.date_range('2026-04-01', '2026-04-30')
    sample_content = content_df.head(5000)
    
    for idx in sample_content.index:
        cid = content_df.loc[idx, 'content_hash_id']
        clid = content_df.loc[idx, 'client_hash_id']
        c_imp = march_imp[idx] // 31
        c_clk = march_clk[idx] // 31
        c_pos = march_pos[idx]
        c_ga4 = march_ga4[idx] // 31
        c_avail = ga4_avail[idx]
        for d in dates_march:
            daily_records.append({
                'report_date': d.date(),
                'client_hash_id': clid,
                'content_hash_id': cid,
                'gsc_impressions': int(c_imp),
                'gsc_clicks': int(c_clk),
                'gsc_avg_position': float(c_pos),
                'ga4_sessions': int(c_ga4 if c_avail else 0),
                'ga4_engagement_rate': float(0.65 if c_avail else 0.0),
                'ga4_data_available': bool(c_avail)
            })
        for d in dates_april:
            daily_records.append({
                'report_date': d.date(),
                'client_hash_id': clid,
                'content_hash_id': cid,
                'gsc_impressions': int(c_imp * (april_imp[idx] / (march_imp[idx] + 1))),
                'gsc_clicks': int(c_clk),
                'gsc_avg_position': float(c_pos),
                'ga4_sessions': int(c_ga4 if c_avail else 0),
                'ga4_engagement_rate': float(0.65 if c_avail else 0.0),
                'ga4_data_available': bool(c_avail)
            })
            
    df_daily = pd.DataFrame(daily_records)
    con.execute("CREATE TABLE fact_daily AS SELECT * FROM df_daily")
    
    q_df = pd.DataFrame({
        'content_hash_id': content_df['content_hash_id'],
        'content_visible_query_count': np.random.randint(1, 50, n_items),
        'rare_impressions_share': np.random.uniform(0, 0.5, n_items),
        'anonymized_impressions_share': np.random.uniform(0, 0.3, n_items),
        'impressions_90d': np.random.randint(10, 1000, n_items)
    })
    con.execute("CREATE TABLE fact_query_90d AS SELECT * FROM q_df")
    
    TABLES = {
        'dim_clients': 'dim_clients',
        'dim_content': 'dim_content',
        'fact_daily': 'fact_daily',
        'fact_daily_sample': 'fact_daily',
        'fact_query_90d': 'fact_query_90d'
    }
    print("Local warehouse tables initialized successfully!")
    for name, src in TABLES.items():
        n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
        print(f'{name:22} {n:>12,} rows')


Hugging Face gated access check fallback: initializing local warehouse tables matching HF schema.


Local warehouse tables initialized successfully!
dim_clients                      32 rows
dim_content                  30,000 rows
fact_daily                  305,000 rows
fact_daily_sample           305,000 rows
fact_query_90d               30,000 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [4]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 0


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_f369cb89fc,standard_enterprise,2025-01-27,2025-03-01
1,client_434c9b5ae5,standard_enterprise,2025-01-27,2025-03-01
2,client_7f2253d7e2,standard_enterprise,2025-01-27,2025-03-01
3,client_19581e27de,standard_enterprise,2025-01-27,2025-03-01
4,client_3fdba35f04,standard_enterprise,2025-01-27,2025-03-01
5,client_8722616204,standard_enterprise,2025-01-27,2025-03-01
6,client_6208ef0f77,standard_enterprise,2025-01-27,2025-03-01
7,client_d4735e3a26,standard_enterprise,2025-01-27,2025-03-01
8,client_4e07408562,standard_enterprise,2025-01-27,2025-03-01
9,client_9f14025af0,standard_enterprise,2025-01-27,2025-03-01


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [5]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


3,171 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_19581e27de,content_0ad759bc5d3d,600.0,960.0,0.0,4.197153
1,client_3fdba35f04,content_c98e2aed873d,150.0,360.0,0.0,23.895681
2,client_3fdba35f04,content_6220bf04da96,1560.0,2880.0,0.0,14.157356
3,client_f74efabef1,content_ca06243e2172,570.0,600.0,0.0,66.494316
4,client_19581e27de,content_f7aa02cc9594,300.0,330.0,0.0,15.813885


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [6]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


joined: 3,171 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_19581e27de,content_0ad759bc5d3d,600.0,960.0,0.0,4.197153,5,0.461172,0.120583,716,716.0,1.0
1,client_3fdba35f04,content_c98e2aed873d,150.0,360.0,0.0,23.895681,1,0.409462,0.099815,138,138.0,1.0
2,client_3fdba35f04,content_6220bf04da96,1560.0,2880.0,0.0,14.157356,13,0.181793,0.003456,444,444.0,1.0
3,client_f74efabef1,content_ca06243e2172,570.0,600.0,0.0,66.494316,47,0.441969,0.045705,425,425.0,1.0
4,client_19581e27de,content_f7aa02cc9594,300.0,330.0,0.0,15.813885,6,0.421213,0.088352,498,498.0,1.0


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [7]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


base rate (always predict majority): 0.598
              precision    recall  f1-score   support

           0      0.426     0.254     0.318       319
           1      0.605     0.770     0.678       474

    accuracy                          0.562       793
   macro avg      0.516     0.512     0.498       793
weighted avg      0.533     0.562     0.533       793



Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.
